# makemore part 2 — MLP (Lecture 3)

Character-level language model after Bengio et al. 2003, following Karpathy's
*Neural Networks: Zero to Hero*, lecture 3.

This notebook is a record of repeated attempts, not a single clean build. Each
attempt below rebuilds the same model a bit further from memory. The live work is
**§6 (23 September)**; everything above it is earlier drafts kept for reference.

| Section | What it covers |
| --- | --- |
| §1 Setup | imports |
| §2 Attempt 1 | data, vocab, the context window |
| §3 Attempt 2 | rebuild + first embedding lookup |
| §4 Attempt 3 | first end-to-end network: params, LR search, minibatch training |
| §5 Attempt 4 | restart from scratch — abandoned partway |
| §6 **23 September** | current attempt: train/dev/test split, training, sampling |
| §7 Reference | 22 September layer-by-layer notes |


## 1. Setup

In [4]:
import torch 
import torch.nn.functional as F 
import matplotlib.pyplot as plt # for making figures
%matplotlib inline 
    

---
## 2. Attempt 1 — data, vocab, and the context window

First pass at the pieces: read `names.txt`, build the character/integer mappings,
then roll a `block_size = 3` context window across each word to produce `X` (context)
and `Y` (next character).

Built on `words[:5]` only, to keep the printout readable while getting the shapes right.

In [5]:
#read in all the words 
words = open('names.txt','r').read().splitlines() #,'r' means read. .read() pulls it as a string. .splitlines() splits string into list item according to line breaks. 
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [6]:
len(words)

32033

In [11]:
#build vocabulary of characters and mappings to/from integers. 
chars = sorted(list(set(''.join(words)))) # sort a list of characters formed from the set of characters in words. 
stoi = {s:i+1 for i,s in enumerate(chars)} #index each character using a dictionary
stoi['.']=0 # assign zero index to .
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [58]:
#build the dataset separate wT and w(T-1) to find some f(w(T-1)) = wT

block_size = 3  # for n=3 in w(T-1)
X,Y = [], []
for w in words[:5]:

    print(w)
    context = [0]*block_size # builds context of [0,0,0] 
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix] 
        
X = torch.tensor(X) #make X a tensor object w ability to calc gradients n stuff. 
Y = torch.tensor(Y) #make Y into a tensor 


emma
olivia
ava
isabella
sophia


---
## 3. Attempt 2 — rebuild, and the first embedding lookup

Same ground again from memory, then one step further: the lookup table `C` that maps
each of the 27 characters to a 2-D vector, and `emb = C[X]` — indexing a tensor with a
tensor of integers, which is what makes the embedding layer a lookup rather than a matmul.

In [31]:
#building vocabulary. this requires us to generate letters --> nos and nos --> letters
words[:3]
chars = sorted(list(set(''.join(words)))) 
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos = {i:s for s,i in stoi.items()}


In [53]:
#build the dataset by creating X and Y which is wT and w(T-n)

block_size = 3 # size of context window, n
X=[]
Y=[]


for w in words[:5]:
    context = [0]*block_size    # to fill initial context window 
    for ch in w+'.':        #for character in word.
        ix= stoi[ch]        #assign index nubmer to character
        X.append(context)   #adds context as a new item in the list of inputs
        Y.append(ix)                  #adds target index to desired product
        context = context[1:]+[ix]

X = torch.tensor(X) #make X a tensor object w ability to calc gradients n stuff. 
Y = torch.tensor(Y) #make Y into a tensor 


In [66]:
#build encoder. Lets build encoder for each letter -> 2d emb(qun: what happens if our encoding is tkaing 3 input integers and encoding it. ik its 27^3 for the dictionary but what difference in performance would it make)
C = torch.randn((27,2)) 

torch.Size([32, 3, 2])

In [67]:
#embedding layer
emb = C[X]     # for every value in X, its indexed by C. 
emb.shape

torch.Size([32, 3, 2])

---
## 4. Attempt 3 — first end-to-end network

The first version that trains. Full dataset this time (no `words[:5]`), a seeded
generator so runs are repeatable, and the whole stack: embedding → hidden layer with
`tanh` → logits → `cross_entropy`.

In [277]:
words = open('names.txt','r').read().splitlines() #splitlines makes it a list according to line breaks

In [285]:
#build the vocab
chars = sorted(list(set(''.join(words)))) # notes that ''.join(words) creates a list. # why wont this work without .join(words) 
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos = {i:s for i,s in stoi.items()}



In [300]:
#build dataset of w(T-1) and wT. inputs and outputs

block_size = 3 #arbitrary number of inputs
X = []
Y = []

for w in words:
    context = [0]*block_size
    for ch in w+'.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:]+ [ix] 

X = torch.tensor(X)
Y = torch.tensor(Y) # make it into tensor objects so we can do later tensor calculations and calculate losses or wtv. These are raw inputs
X.size()   

torch.Size([228146, 3])

### 4.1 Parameters

In [301]:
# Time to build the parameters
g = torch.Generator().manual_seed(2147483647)
C = torch.randn(27,2, generator = g) # 27 letters into 2 dimensional vector encodings. 
W1 = torch.randn(6,100, generator = g) # takes in each example in X (which contains n ints) and multiplies by random set of n weights for each input. 
b1 = torch.randn(100, generator = g)
W2 = torch.randn(100,27, generator = g)
b2 = torch.randn(27, generator = g) # produce 27 logits
parameters = [C,W1,b1,W2,b2] # puts all the tensor scalars into a list for simultaneous update. 

In [302]:
for p in parameters:
    p.requires_grad = True

### 4.2 Learning-rate sweep

A thousand candidate rates spaced evenly in log space between `0.001` and `1`, ready to
step through one per iteration and plot loss against rate.

In [303]:
lre = torch.linspace(-3,0,1000) #1000 numbers between -3 and 0.
lrs = 10**lre                   # from 0.001 to 1.

### 4.3 Minibatch training

32 random examples per step instead of all 228k — noisier gradients, far more steps per
second. The LR-sweep and manual-softmax lines are commented out rather than deleted, so
the alternatives stay visible.

In [327]:
lri = []
lossi = []


for i in range(100):

    #minibatch construct
    ix= torch.randint(0,X.shape[0],(32,)) #randomly select 32 index numbers from total training set
    

    
    #Forward pass
    emb = C[X[ix]]  #encode inputs into embeddings. (32x3x2)
    h = torch.tanh(emb.view(-1,6) @ W1 +b1) # why is this a "hidden layer". how does emb.view know how to manage the torch.size[32,3,2] into [32,6]
    logits = h@W2 +b2 # takes input logits of 100 and outputs vector of dimension 27 for each example. 
    # normalise logits to get probabilities (classification??)
    #counts = logits.exp() #exponentiate to make positive.
    #probs = counts/torch.sum(counts,dim = 1, keepdim= True) #dim = 1 tells it to sum the 1st dimension (sum 27 numbers). keepdim = True makes it (32,1) instead of (32)
    #loss =-torch.log(probs[torch.arange(Y.shape[0]),Y]).mean()
    loss = F.cross_entropy(logits,Y[ix]) #compares unnormalised logits and targets to get loss. (need to be samesize for dim =0 )
    
    
    #Backward pass
    for p in parameters:
        p.grad = None
    loss.backward()#calculate gradients (Backprop)
    
    #update params:
    #lr = lrs[i]
    for p in parameters:
        p.data += -0.01 * p.grad # grad descent 

    #track data
    #lri.append(lre[i])
    #lossi.append(loss.item())
#print(loss.item())

### 4.4 Loss on the full dataset

The training loop above reports loss on its 32-example minibatch, which bounces around.
This runs the forward pass once over everything for a number that can actually be compared
between runs.

In [328]:
#Forward pass
emb = C[X]  #encode inputs into embeddings. (32x3x2)
h = torch.tanh(emb.view(-1,6) @ W1 +b1) # why is this a "hidden layer". how does emb.view know how to manage the torch.size[32,3,2] into [32,6]
logits = h@W2 +b2 # takes input logits of 100 and outputs vector of dimension 27 for each example. 
# normalise logits to get probabilities (classification??)
#counts = logits.exp() #exponentiate to make positive.
#probs = counts/torch.sum(counts,dim = 1, keepdim= True) #dim = 1 tells it to sum the 1st dimension (sum 27 numbers). keepdim = True makes it (32,1) instead of (32)
#loss =-torch.log(probs[torch.arange(Y.shape[0]),Y]).mean()
loss = F.cross_entropy(logits,Y) #compares unnormalised logits and targets to get loss. (need to be samesize for dim =0 )
print(loss.item())

2.3100035190582275


---
## 5. Attempt 4 — from scratch again (abandoned partway)

Another rebuild from memory. It stops mid-line in the parameters cell (`b1 = `), so this
section will raise a `SyntaxError` if run — left exactly as written, since the 23 September
attempt below picks the thread up.

In [1]:
#redoing the implementation above from scratch
words = open('names.txt','r').read().splitlines()
chars = sorted(list(set(''.join(words))))

In [5]:
#building the vocab

stoi= {s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos = {i:s for s,i in stoi.items()} #why do u need the .items()?

In [6]:
# encode dataset. 
X = []
Y = []
context_window = 3 #u can define this. 

for w in words:
    context = [0]*context_window 
    for ch in w+'.':
        ix = stoi[ch]
        Y.append(ix)
        X.append(context)
        context = context[1:] + [ix]

X= torch.tensor(X)
Y= torch.tensor(Y)

In [8]:
#Build embedding (1,2) vector
C = torch.randn((27,2)) #C is a lookup table
#Build parameters
W1 = torch.randn(6,100)
b1 = torch.randn(100)
W2 = torch.randn(100,27)
b2 = torch.randn(27)
params = [C,W1,b1,W2,b2]

In [9]:
#build parameters
C = torch.randn(27,2) # encode each letter
W1 = torch.randn(6,100)
b1 = 

---
## 6. 23 September — current attempt

> `#the blocks below here are attempt on 23 sept:`

The furthest this has gone: a proper train/dev/test split, a wider context window
(`context_window = 4`, 3-D embeddings, so the hidden layer takes `4 × 3 = 12` inputs),
3000 training steps, held-out evaluation, and sampling names back out of the model.

### 6.1 Data

In [26]:
#import dataset
words = open('names.txt','r').read().splitlines()

In [376]:
#build vocab
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.']= 0
itos = {i:s for s,i in stoi.items()}

In [439]:
# build dataset(in numbers)
X = []
Y = []
context_window= 4

for w in words[:50]:
    context = [0]*context_window
    for ch in w+'.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:]+ [ix]
X = torch.tensor(X)
Y = torch.tensor(Y)

### 6.2 Train / dev / test split

`build_dataset` wraps the encoding loop so it can be called three times. The shuffle is
seeded (`random.seed(42)`), so the same names land in the same split on every run.

This supersedes the `words[:50]` cell above — from here on the model trains on `Xtr`/`Ytr`.

In [440]:
#Split into 3 datasets

def build_dataset(words):
    context_window=4
    X,Y = [],[]
    for w in words:

        context = [0]*context_window
        for ch in w+ '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X =torch.tensor(X)
    Y =torch.tensor(Y)
    return X,Y
    
import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,Ytr = build_dataset(words[:n1])
Xdev,Ydev = build_dataset(words[n1:n2])
Xtest,Ytest=build_dataset(words[n2:])


### 6.3 Parameters

`C` is now 27 × 3 rather than 27 × 2, and the context window is 4, so the hidden layer
takes 12 inputs.

In [441]:
# initialise weights.
C = torch.randn(27,3) #encode characters with 3d vector
W1 = torch.randn(12,100)
b1 = torch.randn(100)
W2 = torch.randn(100,27)
b2 = torch.randn(27) 
params= [C,W1,b1,W2,b2]

#turn on gradients for params 
for p in params:
    p.requires_grad = True

### 6.4 Learning-rate candidates

Same log-spaced sweep idea as §4.2, built with `arange` here.

In [442]:
#modify learning rate (not now)
lrs = torch.arange(-3,0 ,0.001)
lre = 10**lrs
lre

tensor([0.0010, 0.0010, 0.0010,  ..., 0.9931, 0.9954, 0.9977])

### 6.5 Training

3000 steps on minibatches of 32 drawn from the training split, at a fixed rate of `0.1`
(chosen from the sweep). `lri`/`lossi` collect the history for plotting.

In [476]:
#training cycles
lri= []
lossi= [] 

for i in range(3000):

    #minibatch construct
    ix = torch.randint(0,Xtr.shape[0],(32,))
    
                      
    #Forward pass
    emb = C[Xtr[ix]] # embed each value of X by appling function C to it. 
    h = torch.tanh(emb.view(-1,12)@W1 + b1) #hidden layer (what is this specifically for or is this just a layer)
    logits = h@W2+b2
    loss = F.cross_entropy(logits,Ytr[ix])
    
    #backward pass
    for p in params:
        p.grad = None      # need to set gradients to nothing so that it stores a value?
    loss.backward()
    for p in params:
        learning_rate =  0.1 #lre[i] originally determined via graph
        p.data -= learning_rate * p.grad

    #record data to determinine optimal learning rate
    lri.append(learning_rate)
    lossi.append(loss.item())
print(loss)


tensor(1.6932, grad_fn=<NllLossBackward0>)


### 6.6 Evaluation on the dev split

The number that matters: loss on names the model never trained on. A dev loss far above
the training loss would mean it is memorising rather than generalising.

In [480]:
#testing
emb = C[Xdev]
h = torch.tanh(emb.view(-1,12) @W1+b1)
logits = h@W2+b2
loss = F.cross_entropy(logits,Ydev)
loss


tensor(2.2519, grad_fn=<NllLossBackward0>)

In [477]:
# use for detecting optimum learning rate: plt.plot(lri,lossi)

### 6.7 Sampling from the model

Feed a context in, get a probability distribution over the next character, sample from it
with `torch.multinomial`, slide the window forward, repeat. The first cell predicts a
single character from four you type; the two after it loop until the model emits `.`
(the end-of-name token) and print a whole name.

In [428]:
#running inference. 
a = input('i predict 5th letter of a name. give 4 letters:')
X_inf=[]
for ch in a:
    X_inf.append(stoi[ch])
emb_inf = C[X_inf]
h_inf = torch.tanh(emb_inf.view(-1,12)@W1 +b1) 
logit_inf = h_inf@W2+b2
softmax_inf = torch.exp(logit_inf)
prob_inf = softmax_inf/torch.sum(softmax_inf,dim = 1, keepdim = True)
itos[torch.multinomial(prob_inf, 1).item()]

i predict 5th letter of a name. give 4 letters: asdf


'e'

In [389]:
# Running inference to write one whole name
next_letter = ''
name = []
a = ['.'] * 4

while next_letter != '.' and len(name) < 10:
    X_inf = []
    
    for ch in a:
        X_inf.append(stoi[ch])

    emb_inf = C[X_inf]
    h_inf = torch.tanh(emb_inf.view(-1, 12) @ W1 + b1)
    logit_inf = h_inf @ W2 + b2
    prob_inf = torch.softmax(logit_inf, dim=1)

    next_letter = itos[torch.multinomial(prob_inf, 1).item()]

    
    
    if next_letter == '.':
        break  # End the name without including the dot

    
    name.append(next_letter)
    a = a[1:] + [next_letter]

print(''.join(name))

hamigail


In [336]:
next_letter = ''
name = []
a = ['.']*4 
while next_letter != '.' and len(name)<10:
    X_inf = []
    for ch in a:
        X_inf.append(stoi[ch])
    emb_inf = C[X_inf]
    h_inf = torch.tanh(emb_inf.view(-1, 12) @ W1 + b1)
    logit_inf = h_inf @ W2 + b2
    
    prob_inf = torch.softmax(logit_inf,dim=1) # dim=1 so that it normalises the 27

    next_letter = itos[torch.multinomial(prob_inf, 1).item()]

    if next_letter == '.':
        break
    name.append(next_letter)
    a= a[1:] + [next_letter]

print(''.join(name))

hare


---
## 7. Reference — 22 September, layer by layer

> `#stuff below here can be discarded(learnings from 22 sept):`

Kept for reference, not part of the current model. This is the network taken apart one
layer at a time — encoder, embedding, hidden layer, output layer, manual softmax — with
`.shape` checked at each step. Superseded by §6, safe to delete.

In [109]:
#build the encoder for the dataset. 
C = torch.randn(27,2) #27 letters into 2 dimensional vector

In [128]:
#since every no. in X is a number from 1-27, it automatically becomes possible for it to act as a lookup?? 
#we apply the fucntion C to all the numbers in X. 
emb = C[X] 
emb.shape

torch.Size([32, 3, 2])

In [111]:
W1 = torch.randn((6,100)) #6 inputs per neuron, 100 neurons. 
b1 = torch.randn(100)     # 100 weights for each neuron

In [153]:
emb[:, 0,:] # emb [no. examples: chars per examples  : embedding coordinates ]
# we want to do this more efficiently: torch.cat((emb[:,0,:],emb[:,1,:],emb[:,2,:]),1)
h = torch.tanh(emb.view(emb.shape[0],6) @ W1 + b1)
h.shape

torch.Size([32, 100])

In [154]:
W2 = torch.randn((100,27))
b2 = torch.randn(27)

In [162]:
logits = h @ W2 + b2 # raw output numbers 
counts = logits.exp() # make positive 
prob = counts/counts.sum(1,keepdims=True)  #for each example, normalise it such that total prob for that example is 1. 
prob.shape 



torch.Size([32, 27])